## Задание 1

Реализуйте класс с полностью инкапсулированным состоянием, используя name mangling и property, обеспечив валидацию при изменении атрибутов и демонстрируя, как Python скрывает "приватные" данные.

In [9]:
class Encapsulated:
    def __init__(self, value):
        self.__value = None
        self.value = value

    @property
    def value(self):
        return self.__value

    @value.setter
    def value(self, new_value):
        if not isinstance(new_value, (int, float)):
            raise ValueError("Значение должно быть числом")
        if new_value < 0:
            raise ValueError("Значение должно быть неотрицательным")
        self.__value = new_value

obj = Encapsulated(10)
print(obj.value)                  # доступ к значению через геттер
print(obj._Encapsulated__value)   # все еще работает потому что не является настоящей приватной переменной

10
10


## Задание 2

Создайте иерархию классов с абстрактным базовым классом (ABC) и абстрактными методами, демонстрируя использование модуля abc. Реализуйте интерфейсы и их проверку через isinstance и issubclass.

In [10]:
from abc import ABC, abstractmethod


class Transport(ABC):
    """Абстрактный класс для транспортных средств."""
    @abstractmethod
    def move(self) -> None:
        """Метод для перемещения транспортного средства."""
        pass

    @abstractmethod
    def capacity(self) -> int:
        """Метод для получения вместимости транспортного средства."""
        pass


class Car(Transport):
    """Конкретный класс, представляющий автомобиль."""
    
    def move(self) -> str:
        return "Машина едет по дороге"

    def capacity(self) -> int:
        return 5


class Airplane(Transport):
    """Конкретный класс, представляющий самолет."""
    
    def move(self) -> str:
        return "Самолет летит по небу"

    def capacity(self) -> int:
        return 180

car = Car()
plane = Airplane()

print(car.move())
print(plane.move())

print(isinstance(car, Transport))       # True
print(issubclass(Car, Transport))       # True

try:
    t = Transport()
except TypeError as e:
    print("can't instantiate abstract class")

Машина едет по дороге
Самолет летит по небу
True
True
can't instantiate abstract class


## Задание 3

Реализуйте дженерик-функцию (duck typing) с использованием протоколов (PEP 544) и типовых подсказок, чтобы показать полиморфизм без наследования.

In [12]:
from typing import Protocol


class Drawable(Protocol):
    """Класс для определения интерфейса для объектов, которые могут быть нарисованы."""
    def draw(self) -> None:
        ...


class Circle:
    """Конкретный класс, представляющий круг."""
    def draw(self) -> None:
        print("Рисуем круг")


class Square:
    """Конкретный класс, представляющий квадрат."""
    def draw(self) -> None:
        print("Рисуем квадрат")

# функция, которая принимает объект, реализующий интерфейс Drawable, и вызывает его метод draw
def paint(obj: Drawable):
    obj.draw()

# функция paint может работать с любым объектом, который реализует метод draw
paint(Circle())
paint(Square())

Рисуем круг
Рисуем квадрат


## Задание 4

Опишите и продемонстрируйте работу метода getattribute в отличие от getattr, реализуйте логирование всех обращений к атрибутам объекта, исследуйте возможные зацикливания.

In [13]:
class Logger:

    def __getattribute__(self, name):
        """Вызывается при доступе к любому атрибуту (существующему или нет)."""
        print(f"__getattribute__ вызван для: {name}")
        
        try:
            # для избегания бесконечной рекурсии, используем базовый метод __getattribute__
            value = object.__getattribute__(self, name)
            print(f"Найден атрибут '{name}' со значением: {value}")
            return value
        except AttributeError:
            print(f"Атрибут '{name}' не найден, делегируем вызов __getattr__")
            raise # пробрасываем исключение, чтобы __getattr__ мог его обработать

    def __getattr__(self, name):
        """Вызывается только для отсутствующих атрибутов."""
        print(f"__getattr__ вызван для отсутствующего атрибута: {name}")
        return f"Значение по умолчанию для {name}"


obj = Logger()
obj.x = 5

print(obj.x)
print(obj.y)

__getattribute__ вызван для: x
Найден атрибут 'x' со значением: 5
5
__getattribute__ вызван для: y
Атрибут 'y' не найден, делегируем вызов __getattr__
__getattr__ вызван для отсутствующего атрибута: y
Значение по умолчанию для y


## Задание 5

Создайте класс, который использует метакласс, объясните как метаклассы влияют на создание классов в Python, и реализуйте контролируемое изменение класса через метакласс.

In [ ]:
class Meta(type):
    """Метакласс для демонстрации создания класса и изменения его атрибутов."""

    def __new__(mcs, name, bases, namespace):
        print(f"Создание класса: {name}")

        # добавим новый атрибут
        namespace["added_by_meta"] = "Привет от метакласса!"

        # изменим имя класса
        new_name = "Good" + name

        # вызовем базовый метод __new__ для создания класса
        return super().__new__(mcs, new_name, bases, namespace)


class GoodClass(metaclass=Meta):
    pass

'''
Метаклассы позволяют нам контролировать процесс создания классов. Они могут изменять
атрибуты класса, добавлять новые методы или даже изменять имя класса. Здесь мы создали
метакласс Meta, который добавляет новый атрибут и изменяет имя класса при его создании.
Когда мы создаем класс GoodClass с помощью этого метакласса, он автоматически получает
новый атрибут и измененное имя.
'''

Создание класса: GoodClass


## Задание 6

Реализуйте класс с дескрипторами данных и неданных, объясните разницу между ними и механизм вызова методов get, set, delete, особенно при наследовании.

In [18]:
class DataDescriptor:
    def __get__(self, instance, owner):
        print("DataDescriptor __get__")
        return instance.__dict__.get("_data", None)

    def __set__(self, instance, value):
        print("DataDescriptor __set__")
        instance.__dict__["_data"] = value

    def __delete__(self, instance):
        print("DataDescriptor __delete__")
        del instance.__dict__["_data"]


class NonDataDescriptor:
    def __get__(self, instance, owner):
        print("NonDataDescriptor __get__")
        return "Non-data descriptor value"


class MyClass2:
    data_desc = DataDescriptor()
    non_data_desc = NonDataDescriptor()


obj = MyClass2()
obj.data_desc = 10
print(obj.data_desc)
print(obj.non_data_desc)

'''
Data Descriptor - это дескриптор, который реализует методы __get__, __set__ и __delete__.
Он управляет доступом к атрибуту и может изменять его значение.
Non-Data Descriptor - это дескриптор, который реализует только метод __get__.
Он не может изменять значение атрибута, а только возвращать его.

При наследовании, если класс имеет атрибут с именем, совпадающим с именем дескриптора,
то атрибут класса будет иметь приоритет над дескриптором. Это означает, что если мы
определим атрибут data_desc в классе MyClass2, то он будет использоваться вместо метода
__get__ и __set__ из DataDescriptor. Однако, если мы не определим атрибут data_desc в
MyClass2, то при доступе к data_desc будет вызываться метод __get__ из DataDescriptor,
а при присваивании значения - метод __set__.
'''

DataDescriptor __set__
DataDescriptor __get__
10
NonDataDescriptor __get__
Non-data descriptor value


'\nData Descriptor - это дескриптор, который реализует методы __get__, __set__ и __delete__.\nОн управляет доступом к атрибуту и может изменять его значение.\nNon-Data Descriptor - это дескриптор, который реализует только метод __get__.\nОн не может изменять значение атрибута, а только возвращать его.\n\nПри наследовании, если класс имеет атрибут с именем, совпадающим с именем дескриптора,\nто атрибут класса будет иметь приоритет над дескриптором. Это означает, что если мы\nопределим атрибут data_desc в классе MyClass2, то он будет использоваться вместо метода\n__get__ и __set__ из DataDescriptor. Однако, если мы не определим атрибут data_desc в\nMyClass2, то при доступе к data_desc будет вызываться метод __get__ из DataDescriptor,\nа при присваивании значения - метод __set__.\n'

## Задание 7

Напишите класс с поддержкой множественного наследования, демонстрирующий работу C3-линеаризации MRO на сложном примере с 3+ уровнями наследования и пересечениями.

In [ ]:
class A:
    def method(self):
        print("A.method")


class B(A):
    def method(self):
        print("B.method")
        super().method()


class C(A):
    def method(self):
        print("C.method")
        super().method()


class D(B, C):
    def method(self):
        print("D.method")
        super().method()


d = D()
d.method()
print(D.__mro__)

'''
При вызове метода method() на экземпляре класса D, Python использует алгоритм
MRO (Method Resolution Order) для определения порядка поиска методов в иерархии
классов. В данном случае, порядок будет следующим: D -> B -> C -> A. Поэтому при
вызове d.method(), будет выполнен метод D.method(), затем B.method(), затем
C.method(), затем A.method().
'''

D.method
B.method
C.method
A.method
(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)


## Задание 8

Реализуйте класс с пользовательскими dunder-методами: new, init, call, del, repr, str, и объясните, как Python вызывает их в цикле жизни объекта.

In [ ]:
class LifeCycle:
    
    def __new__(cls, *args, **kwargs):
        print("__new__ called")
        instance = super().__new__(cls)
        return instance

    def __init__(self):
        print("__init__ called")

    def __str__(self):
        return "String representation (__str__)"

    def __repr__(self):
        return "Repr representation (__repr__)"

    def __call__(self):
        print("__call__ called")

    def __del__(self):
        print("__del__ called")


obj = LifeCycle()
print(obj)
obj()
del obj

'''
Метод __new__ вызывается при создании нового экземпляра класса и отвечает за
выделение памяти для объекта. Он возвращает новый экземпляр класса.
Метод __init__ вызывается после создания экземпляра и отвечает за его инициализацию.
Метод __str__ возвращает строковое представление объекта для удобного чтения.
Метод __repr__ возвращает строковое представление объекта для разработчика, обычно  используемое для отладки.
Метод __call__ позволяет экземпляру класса быть вызываемым как функция.
Метод __del__ вызывается при удалении объекта и может использоваться для
освобождения ресурсов или выполнения других действий при уничтожении объекта.
'''

__new__ called
__init__ called
String representation (__str__)
__call__ called
__del__ called


## Задание 9

Создайте контекстный менеджер с помощью специальных методов enter и exit, используйте его вместе с классом, в котором присутствуют методы с разграничением прав доступа.

https://habr.com/ru/articles/739326/

In [22]:
class Access:
    def __init__(self):
        self._secret = "password123"
        self._locked = True

    def __enter__(self):
        print("__enter__: доступ разрешен")
        self._locked = False
        return self._secret

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("__exit__: доступ закрыт")
        self._locked = True
        return False

    @property
    def secret(self):
        if not self._locked:
            return self._secret
        return "ДОСТУП ЗАПРЕЩЕН"

obj = Access()
print(obj.secret)
with obj as secret_value:
    print(secret_value)
print(obj.secret)

ДОСТУП ЗАПРЕЩЕН
__enter__: доступ разрешен
password123
__exit__: доступ закрыт
ДОСТУП ЗАПРЕЩЕН


## Задание 10

Создайте класс, объекты которого могут быть отслежены с помощью слабых ссылок (weakref). Реализуйте систему, которая хранит слабые ссылки на все созданные объекты класса и автоматически удаляет их из списка при уничтожении объектов. Продемонстрируйте это поведение, выводя текущее количество живых объектов.

In [23]:
import weakref


class TrackedObject:
    _instances = []

    def __init__(self, name):
        self.name = name

        # создаём weak reference с callback
        self_ref = weakref.ref(self, TrackedObject._remove_instance)
        TrackedObject._instances.append(self_ref)

        print(f"Создан: {self.name}")

    @classmethod
    def _remove_instance(cls, weak_ref):
        print("Удалён объект, удаляем его из списка")
        cls._instances = [r for r in cls._instances if r is not weak_ref]

    @classmethod
    def live_count(cls):
        # чистим мертвые ссылки на всякий случай
        cls._instances = [r for r in cls._instances if r() is not None]
        return len(cls._instances)

    def __del__(self):
        print(f"__del__ вызван для {self.name}")

a = TrackedObject("A")
b = TrackedObject("B")
c = TrackedObject("C")

print("Живых объектов:", TrackedObject.live_count())

del b

print("После удаления b:", TrackedObject.live_count())

Создан: A
Создан: B
Создан: C
Живых объектов: 3
__del__ вызван для B
Удалён объект, удаляем его из списка
После удаления b: 2
